## Import libraries and data

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import csv
import xml.etree.ElementTree as ET
from Tool import clean_speed, fill_tracks, clean_elec, from_df_to_geodf, neighbour_gauge, final_gauge, haversine_distance

## Inputs

In [ ]:
#Change inputs here if needed
rail_data = "data/uk_railways_osm.gpkg"
Train_Planning_file = "data/TrainPlanningRules.xml"
Stops = "data/Stops.csv"
loading_gauge = "data/loading_gauge.geojson"
nodes_file = "nodes.csv"

## Import of the datasets and conversion into a csv file

In [ ]:
# 1. Chargement du fichier
gdf = gpd.read_file(rail_data)

# 2. Conservation de la géométrie
# On transforme la géométrie en format WKT (Well-Known Text).
# C'est un format texte que le CSV accepte et qui contient tous les points GPS.
# Exemple : "LINESTRING (-1.81 52.57, -1.81 52.56)"
gdf['geometry_wkt'] = gdf['geometry'].apply(lambda x: x.wkt)

# 3. Extraction des coordonnées de départ et d'arrivée (pour tes noeuds/jonctions)
# C'est très utile pour tes futurs calculs de graphe dans Excel
gdf['start_lon'] = gdf['geometry'].apply(lambda x: x.coords[0][0])
gdf['start_lat'] = gdf['geometry'].apply(lambda x: x.coords[0][1])
gdf['end_lon'] = gdf['geometry'].apply(lambda x: x.coords[-1][0])
gdf['end_lat'] = gdf['geometry'].apply(lambda x: x.coords[-1][1])

# 4. Nettoyage de la colonne 'other_tags'
# Dans OSM, cette colonne contient souvent des données clés sous forme de texte 
# (ex: "gauge"=>"1435"). On va la garder telle quelle, mais assure-toi qu'elle est bien lue.
if 'other_tags' in gdf.columns:
    gdf['other_tags'] = gdf['other_tags'].astype(str)

# 5. Suppression de la colonne géométrie originale (non compatible CSV)
df_rail = pd.DataFrame(gdf.drop(columns=['geometry']))

# 6. Export en CSV
# J'utilise le séparateur ";" car les noms de gares contiennent souvent des virgules ","
df_rail.to_csv("complete_railway_network.csv", index=False, sep=';', encoding='utf-8')

print(f"Export terminé ! {len(df_rail)} lignes exportées.")

In [ ]:
df_rail.head()

## Cleaning of the dataset

### Selection of only national Railway parts

In [ ]:
# Selection of only usable national Railway parts
df_rail = df_rail[df_rail['railway'] == 'rail']
df_rail = df_rail[df_rail['usage'].isin(['main', 'branch', 'industrial', 'none', None])]

In [ ]:
len(df_rail)

Clean speed format and fill missing values with 60 mph.

In [ ]:
df_rail = clean_speed(df_rail)

## Fill the missing values in terms of track numbers

Here, we consider to fill the branch or industrial types with 1 track, the others with 2 tracks.

In [ ]:
df_rail = fill_tracks(df_rail)
df_rail["tracks"] = df_rail["tracks"].astype(int)

We will simplify different type of electrification

In [ ]:
df_rail['elec_type'] = df_rail.apply(clean_elec, axis=1)

In [ ]:
df_rail.columns

In [ ]:
to_drop = [
    "highway", "waterway", "aerialway", "barrier", "man_made",
    "railway_aws", "railway_tpws", "ref", "layer", "foot", "lit",
    "segregated", "smoothness", "surface", "width", "bicycle",
    "z_order", "other_tags"
]

df_rail = df_rail.drop(columns=[c for c in to_drop if c in df_rail.columns])

In [ ]:
# Affiche chaque valeur unique une seule fois
print("Valeurs uniques dans 'electrified':")
print(df_rail['electrified'].unique())

# Affiche le nombre d'apparitions de chaque valeur (très utile pour voir les erreurs)
print("\nStatistiques de la colonne 'electrified':")
print(df_rail['electrified'].value_counts(dropna=False))

In [ ]:
df_rail.head()

In [ ]:
print("Valeurs uniques dans Usage:")
print(df_rail['tracks'].unique())
print(df_rail['tracks'].value_counts(dropna=False))

In [ ]:
df_rail = df_rail.drop_duplicates(subset=["start_lon", "start_lat", "end_lon", "end_lat"])
df_rail = df_rail[~df_rail["usage"].isin(["industrial", "military"])]


In [ ]:
len(df_rail)

## Load the loading gauge dataset, clean it and join it with the previous dataset

In [ ]:
#Load datasets

gdf_rail = from_df_to_geodf(df_rail)
gdf_loading_gauge = gpd.read_file(loading_gauge)

#Join and check the neighbours to apply to missing loading gauges
df_complete = neighbour_gauge(gdf_rail, gdf_loading_gauge)

#If no neighboutrs, check the voltage to find the loading gauge
df_complete['loading_gauge_final'] = df_complete.apply(final_gauge, axis=1)

#Spread of different loading gauges
print(df_complete['loading_gauge_final'].value_counts())

### Outliers : there are "W1" gauges that need to be removed

In [ ]:
#We replace these W1 values by W6 values
df_complete.loc[df_complete['loading_gauge_final'] < 6, 'loading_gauge_final'] = 6
print(df_complete['loading_gauge_final'].value_counts().sort_index())

df_complete.to_csv("Final_dataframe_rail.csv", index=False)

In [ ]:
df_complete = df_complete.drop(columns=['railway', 'index_right', 'loading_gauge_val'])

## Add the distance between two nodes

In [ ]:
df_complete['distance'] = haversine_distance(df_complete["start_lat"],df_complete["start_lon"],df_complete["end_lat"],df_complete["end_lon"])
df_complete['min_travel_time'] = df_complete['distance'] / df_complete['maxspeed']

In [ ]:
df_complete.head()

## Upload a file with names and location for train stations

In [ ]:
with open(nodes_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "tiploc", "name"])

    for event, elem in ET.iterparse(Train_Planning_file, events=("end",)):
        if elem.tag == "TTPRLocation":
            
            writer.writerow([
                elem.attrib.get("Id"),
                elem.attrib.get("Tiploc"),
                elem.attrib.get("Description")
            ])
            
            elem.clear()

In [ ]:
df_nodes = pd.read_csv(nodes_file)

#Delete row without ID
df_nodes = df_nodes.dropna(subset=['id'])
df_nodes = df_nodes.drop_duplicates()

#Clean the str and make sure any blank space is there
df_nodes['tiploc'] = df_nodes['tiploc'].astype(str).str.strip()
df_nodes['name'] = df_nodes['name'].astype(str).str.strip()

invalid_values = ["unknown", "none", "nan"]

df_nodes = df_nodes[
    ~df_nodes.drop(columns=["id"])
      .apply(lambda col: col.astype(str).str.strip().str.lower())
      .isin(invalid_values)
      .all(axis=1)
]


df_nodes.to_csv("nodes_clean.csv", index=False)

print("Number of nodes remaining : ", len(df_nodes))

In [ ]:
df_nodes.head()

In [ ]:
df_lon_lat = pd.read_csv("data/stations.csv")

print("Number of nodes remaining (nodes) : ", len(df_nodes))
print("Number of nodes remaining (lon_lat) : ", len(df_lon_lat))

df_name_loc = pd.merge(
    df_nodes, 
    df_lon_lat, 
    left_on='name', 
    right_on='stationName', 
    how='left'
)

df_name_loc.head(10)

In [ ]:
print(len(df_name_loc))
df_name_loc = df_name_loc.drop(columns=['stationName', 'iataAirportCode', 'constituentCountry'])

In [ ]:
df_name_loc.head()

## Finally, we join df_name_loc with df_complete to have a final qualitative dataset

In [ ]:
print(df_name_loc.columns)
print(df_complete.columns)

# Graph creation

In [ ]:
G = nx.Graph()

for _, row in df_complete.iterrows():
    start = (row.start_lon, row.start_lat)
    end = (row.end_lon, row.end_lat)
    G.add_edge(start, end)


In [ ]:
components = list(nx.connected_components(G))
print("Nombre de composantes :", len(components))


In [ ]:
largest = max(components, key=len)
print("Taille de la plus grande composante :", len(largest))


In [ ]:
pos = {node: (node[0], node[1]) for node in G.nodes()}
nx.draw(G, pos, node_size=1, edge_color="gray")


In [ ]:
#Quick diagnostic
nodes_with_edges = G.number_of_nodes()
print("===== Network diagnostic ===== \n")
print(f"Number of edges : {G.number_of_edges()}")
print(f"Number of connected nodes : {G.number_of_nodes()}")

#Connectivity checks

if nx.is_connected(G):
    print("Network is fully connected.")
else:
    components = sorted(nx.connected_components(G), key=len, reverse=True)
    giant_size = len(components[0])
    print(f"Network fragmented in {len(components)} parts.")
    print(f"Giant Component : {giant_size} nodes")
    print(f"Coverage ratio : {(giant_size / nodes_with_edges)*100:.1f}%")

In [ ]:
sizes = sorted([len(c) for c in components], reverse=True)
print(sizes[:20])

In [ ]:
df_rail = df_rail[~((df_rail.start_lon == df_rail.end_lon) & (df_rail.start_lat == df_rail.end_lat))]

In [ ]:
#Création d'un autre graphe avec le giant component uniquement
components = list(nx.connected_components(G))
giant = max(components, key=len)

G_giant = G.subgraph(giant).copy()
df_giant = df_rail[
    df_rail.apply(
        lambda r: (r.start_lon, r.start_lat) in giant
               and (r.end_lon, r.end_lat) in giant,
        axis=1
    )
].copy()

pos_giant = {node: (node[0], node[1]) for node in G_giant.nodes()}
nx.draw(G_giant, pos_giant, node_size=1, edge_color="gray")

In [ ]:
print("Original nodes :", G.number_of_nodes())
print("Original edges :", G.number_of_edges())

print("Giant nodes :", G_giant.number_of_nodes())
print("Giant edges :", G_giant.number_of_edges())

ratio_nodes = G_giant.number_of_nodes() / G.number_of_nodes() * 100
ratio_edges = G_giant.number_of_edges() / G.number_of_edges() * 100

print(f"Node coverage : {ratio_nodes:.1f}%")
print(f"Edge coverage : {ratio_edges:.1f}%")


In [ ]:
print("Original components :", nx.number_connected_components(G))
print("Giant components :", nx.number_connected_components(G_giant))

In [ ]:
pos = {node: (node[0], node[1]) for node in G.nodes()}
nx.draw(G, pos, node_size=1, edge_color="gray")

In [ ]:
pos_giant = {node: (node[0], node[1]) for node in G_giant.nodes()}
nx.draw(G_giant, pos_giant, node_size=1, edge_color="gray")

## Graph 2

In [ ]:
G = nx.Graph()

for _, row in df_name_loc.iterrows():

    node_id = row["tiploc"] if pd.notna(row["tiploc"]) else row["id"]

    G.add_node(
        node_id,
        type="station",
        name=row["name"],
        lat=row["lat"],
        lon=row["long"],
        crs=row["crsCode"]
    )

for _, row in df_complete.iterrows():

    start = (row.start_lon, row.start_lat)
    end = (row.end_lon, row.end_lat)

    G.add_node(start, type="osm_point", lon=row.start_lon, lat=row.start_lat)
    G.add_node(end, type="osm_point", lon=row.end_lon, lat=row.end_lat)

    G.add_edge(
        start,
        end,
        distance=row["distance"],
        maxspeed=row["maxspeed"],
        loading_gauge=row["loading_gauge_final"],
        electrified=row["elec_type"],
        gauge=row["gauge"],
        tracks=row["tracks"],
        usage=row["usage"]
    )


#We merge the two graphs
osm_nodes = np.array([node for node in G.nodes if isinstance(node, tuple)])

for _, row in df_name_loc.iterrows():

    station_id = row["tiploc"] if pd.notna(row["tiploc"]) else row["id"]
    station_point = np.array([row["long"], row["lat"]])

    #We aggregagte the nearest osm node to the tnode station
    dists = np.linalg.norm(osm_nodes - station_point, axis=1)
    nearest = tuple(osm_nodes[np.argmin(dists)])

    G.add_edge(station_id, nearest, type="station_link")


In [ ]:
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())


In [ ]:
stations = [n for n, d in G.nodes(data=True) if d.get("type") == "station"]
print("Stations:", len(stations))
stations[:10]


In [ ]:
for s in stations[:10]:  # on teste les 20 premières
    print(s, "degree =", G.degree(s))


In [ ]:
isolated = list(nx.isolates(G))
isolated


In [ ]:
edges = [(u, v, d) for u, v, d in G.edges(data=True) if d.get("type") != "station_link"]
edges[0]



In [ ]:
import pandas as pd

distances = [d["distance"] for _,_,d in G.edges(data=True) if "distance" in d]
pd.Series(distances).describe()


In [ ]:
components = sorted(nx.connected_components(G), key=len, reverse=True)
len(components), len(components[0])


In [ ]:
path = nx.shortest_path(G, source=stations[0], target=stations[1], weight="distance_miles")
len(path)


## Keep the main component of the graph

In [ ]:
# Trouver la plus grande composante
main = max(nx.connected_components(G), key=len)

# Créer un sous-graphe propre
G_main = G.subgraph(main).copy()

print("Nodes:", G_main.number_of_nodes())
print("Edges:", G_main.number_of_edges())


In [ ]:
import matplotlib.pyplot as plt

pos = {n: (d["lon"], d["lat"]) for n, d in G_main.nodes(data=True) if "lon" in d}

plt.figure(figsize=(12, 12))
nx.draw(G_main, pos, node_size=1, edge_color="red")
plt.show()


In [ ]:
import folium

# Centrer la carte sur le Royaume-Uni
m = folium.Map(location=[54, -2], zoom_start=6)

# Ajouter les arêtes
for u, v, d in G_main.edges(data=True):
    if isinstance(u, tuple) and isinstance(v, tuple):
        folium.PolyLine(
            locations=[(u[1], u[0]), (v[1], v[0])],
            color="blue",
            weight=2,
            opacity=0.6
        ).add_to(m)

# Ajouter les stations
for n, d in G_main.nodes(data=True):
    if d.get("type") == "station":
        lat = d.get("lat")
        lon = d.get("lon")
        if lat is None or lon is None:
            continue  # on ignore les stations sans coordonnées
        if pd.isna(lat) or pd.isna(lon):
            continue  # on ignore les NaN

        folium.CircleMarker(
            location=[lat, lon],
            radius=4,
            color="red",
            fill=True
        ).add_to(m)

m
